# City-Level 80/20 Holdout Evaluation

Self-contained experiment: same preprocessing and models as the main gun violence ML pipeline, but with a **city-level** train/test split (~80% / ~20% hexes, seed=42). No city appears in both sets.

**Rules for this folder**
- Reads ../Master_Hexagon_Gun_Violence_Summary_Table.csv only (does not modify any main project files)
- Writes all artifacts under ./output/


In [ ]:
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

BASE_DIR = Path('.').resolve()
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Read-only path to the master CSV in the project root (two levels up)
DATA_PATH = BASE_DIR.parents[1] / 'Master_Hexagon_Gun_Violence_Summary_Table.csv'
RANDOM_STATE = 42
TEST_FRAC = 0.20

print('BASE_DIR   :', BASE_DIR)
print('DATA_PATH  :', DATA_PATH)
print('OUTPUT_DIR :', OUTPUT_DIR)
print('Exists?    :', DATA_PATH.exists())


## 1. Load data


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Cities: {df["City_name"].nunique()}')
print(f'Target > 0 rate: {(df["Total_Gun_Violence"] > 0).mean():.3f}')
print(df.head(3))


## 2. Same feature drops as main pipeline (keep City_name for the split)


In [ ]:
cheat_features = [
    'Accidental_Death', 'Accidental_Injuries', 'Children_Injured',
    'Children_Killed', 'Mass_Shootings', 'Teens_injured', 'Teens_Killed', 'Bars'
]
geometric_features = ['Shape_Length', 'Shape_Area']
admin_features = [
    'QA_Flag', 'aggregationMethod', 'HasData', 'ORIGINAL_OID',
    'sourceCountry', 'apportionmentConfidence', 'populationToPolygonSizeRating'
]
# Main pipeline also drops City_name as a geographic feature for modeling.
# We KEEP it only until after the city holdout split, then drop it from features.
features_to_drop = cheat_features + geometric_features + admin_features
existing_drops = [f for f in features_to_drop if f in df.columns]

df_clean = df.drop(columns=existing_drops)
print(f'Dropped {len(existing_drops)} non-predictive columns')
print(f'Shape after drop: {df_clean.shape}')


## 3. Demographic percentage features (same as main pipeline)


In [ ]:
df_pct = df_clean.copy()
total_pop = df_pct['Total_Population'].replace(0, np.nan)
created_percentage_features = []

if 'Black_Population' in df_pct.columns:
    df_pct['Black_Percentage'] = (df_pct['Black_Population'] / total_pop * 100).fillna(0)
    created_percentage_features.append('Black_Percentage')
if 'White_Population' in df_pct.columns:
    df_pct['White_Percentage'] = (df_pct['White_Population'] / total_pop * 100).fillna(0)
    created_percentage_features.append('White_Percentage')
if 'Population_Below_Poverty_Level' in df_pct.columns:
    df_pct['Poverty_Rate_Pct'] = (df_pct['Population_Below_Poverty_Level'] / total_pop * 100).fillna(0)
    created_percentage_features.append('Poverty_Rate_Pct')

if 'Total_Educational_Attainment_Base' in df_pct.columns:
    edu_base = df_pct['Total_Educational_Attainment_Base'].replace(0, np.nan)
    if 'Education_Less_Than_Ninth_Grade' in df_pct.columns:
        df_pct['Education_Less_Than_9th_Pct'] = (df_pct['Education_Less_Than_Ninth_Grade'] / edu_base * 100).fillna(0)
        created_percentage_features.append('Education_Less_Than_9th_Pct')
    if 'Education_Max_Have_High_School_Diploma' in df_pct.columns:
        df_pct['Education_HS_Diploma_Pct'] = (df_pct['Education_Max_Have_High_School_Diploma'] / edu_base * 100).fillna(0)
        created_percentage_features.append('Education_HS_Diploma_Pct')
    if 'Education_Have_Bachelors_Degree' in df_pct.columns:
        df_pct['Education_Bachelors_Pct'] = (df_pct['Education_Have_Bachelors_Degree'] / edu_base * 100).fillna(0)
        created_percentage_features.append('Education_Bachelors_Pct')

raw_count_features_to_remove = [
    'Black_Population', 'White_Population', 'Population_Below_Poverty_Level',
    'Education_Less_Than_Ninth_Grade', 'Education_Max_Have_High_School_Diploma',
    'Education_Have_Bachelors_Degree', 'Total_Educational_Attainment_Base'
]
existing_raw = [c for c in raw_count_features_to_remove if c in df_pct.columns]
df_pct = df_pct.drop(columns=existing_raw)

print('Created percentage features:', created_percentage_features)
print('Removed raw counts:', existing_raw)
print('Shape:', df_pct.shape)


## 4. City-level ~80/20 hex split (seed=42, no city in both sets)

Algorithm: shuffle cities with RandomState(42), then choose the prefix cut whose cumulative hex count is closest to 20% of all hexes.


In [ ]:
def city_hex_holdout_split(frame, city_col='City_name', test_frac=0.20, seed=42):
    sizes = frame.groupby(city_col).size()
    cities = sizes.index.to_numpy().copy()
    rng = np.random.RandomState(seed)
    rng.shuffle(cities)

    total = len(frame)
    ordered_counts = np.array([sizes[c] for c in cities])
    cum = np.cumsum(ordered_counts)
    fracs = cum / total
    cut = int(np.argmin(np.abs(fracs - test_frac)))
    test_cities = set(cities[: cut + 1])
    train_cities = set(cities[cut + 1 :])

    assign = frame[city_col].map(lambda c: 'test' if c in test_cities else 'train')
    return assign, sorted(train_cities), sorted(test_cities)

split_label, train_cities, test_cities = city_hex_holdout_split(
    df_pct, city_col='City_name', test_frac=TEST_FRAC, seed=RANDOM_STATE
)

city_assignment = df_pct.groupby('City_name').size().reset_index(name='n_hexes')
city_assignment['split'] = city_assignment['City_name'].apply(
    lambda c: 'test' if c in set(test_cities) else 'train'
)
city_assignment = city_assignment.sort_values(['split', 'City_name']).reset_index(drop=True)

n_train = int((split_label == 'train').sum())
n_test = int((split_label == 'test').sum())
print(f'Train cities ({len(train_cities)}): {train_cities}')
print(f'Test cities  ({len(test_cities)}): {test_cities}')
print(f'Train hexes: {n_train:,} ({n_train/len(df_pct)*100:.1f}%)')
print(f'Test hexes:  {n_test:,} ({n_test/len(df_pct)*100:.1f}%)')
assert set(train_cities).isdisjoint(set(test_cities))
print(city_assignment.to_string(index=False))

city_assignment_path = OUTPUT_DIR / 'city_assignment.csv'
city_assignment.to_csv(city_assignment_path, index=False)
print('Saved:', city_assignment_path)


## 5. Min-Max scaling (fit on train only) + binary target


In [ ]:
preserve_cols = ['GRID_ID', 'City_name', 'Total_Gun_Violence']
numeric_cols = df_pct.select_dtypes(include=[np.number]).columns.tolist()
cols_to_scale = [c for c in numeric_cols if c not in ['Total_Gun_Violence']]

train_mask = split_label == 'train'
test_mask = split_label == 'test'

scaler = MinMaxScaler(feature_range=(0, 1))
scaler.fit(df_pct.loc[train_mask, cols_to_scale])

scaled_all = pd.DataFrame(
    scaler.transform(df_pct[cols_to_scale]),
    columns=cols_to_scale,
    index=df_pct.index
)
df_scaled = pd.concat([df_pct[preserve_cols], scaled_all], axis=1)

y_binary = (df_scaled['Total_Gun_Violence'] > 0).astype(int)
print(f'Scaled {len(cols_to_scale)} features using TRAIN statistics only')
print(f'Overall positive rate: {y_binary.mean():.3f}')
print(f'Train positive rate:   {y_binary[train_mask].mean():.3f}')
print(f'Test positive rate:    {y_binary[test_mask].mean():.3f}')


## 6. Iterative VIF pruning (train only, threshold ≤ 5) + pairwise |r| ≥ 0.75 cleanup


In [ ]:
def calculate_vif(frame):
    vif_data = pd.DataFrame()
    vif_data['Feature'] = frame.columns
    vif_data['VIF'] = [variance_inflation_factor(frame.values, i) for i in range(frame.shape[1])]
    return vif_data.sort_values('VIF', ascending=False)

feature_cols_all = [c for c in df_scaled.columns if c not in preserve_cols]
X_train_raw = df_scaled.loc[train_mask, feature_cols_all].copy()

df_vif_clean = X_train_raw.copy()
removed_vif = []
iteration = 0
print('Starting iterative VIF pruning on TRAIN...')
while True:
    iteration += 1
    vif_current = calculate_vif(df_vif_clean)
    max_vif = vif_current['VIF'].max()
    if max_vif <= 5:
        print(f'Convergence: all VIF <= 5 after {iteration - 1} removals')
        break
    worst_feature = vif_current.iloc[0]['Feature']
    worst_vif = float(vif_current.iloc[0]['VIF'])
    print(f'Iteration {iteration}: remove {worst_feature} (VIF={worst_vif:.2f})')
    df_vif_clean = df_vif_clean.drop(columns=[worst_feature])
    removed_vif.append(worst_feature)
    if iteration > 100:
        print('Stopped after 100 iterations')
        break

# Pairwise correlation on train
corr_threshold = 0.75
corr_matrix = df_vif_clean.corr()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
features_to_drop_corr = []
y_train_cont = df_scaled.loc[train_mask, 'Total_Gun_Violence']
df_with_target = df_vif_clean.copy()
df_with_target['Total_Gun_Violence'] = y_train_cont.values
target_corr = df_with_target.corr()['Total_Gun_Violence'].drop('Total_Gun_Violence')

for column in upper_tri.columns:
    high_corr_features = upper_tri[column][abs(upper_tri[column]) >= corr_threshold].index.tolist()
    for feature in high_corr_features:
        corr1 = abs(target_corr[column])
        corr2 = abs(target_corr[feature])
        drop_feat = feature if corr1 >= corr2 else column
        if drop_feat not in features_to_drop_corr:
            features_to_drop_corr.append(drop_feat)
            print(f'Correlation drop: {drop_feat} (|r|>= {corr_threshold})')

final_features = [c for c in df_vif_clean.columns if c not in features_to_drop_corr]
print(f'Final feature count: {len(final_features)}')
print(final_features)

X_train = df_scaled.loc[train_mask, final_features].copy()
X_test = df_scaled.loc[test_mask, final_features].copy()
y_train = y_binary.loc[train_mask].copy()
y_test = y_binary.loc[test_mask].copy()
grid_test = df_scaled.loc[test_mask, 'GRID_ID'].copy()
city_test = df_scaled.loc[test_mask, 'City_name'].copy()

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')


## 7. Train models (same hyperparameters as main pipeline)


In [ ]:
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, max_depth=15, random_state=RANDOM_STATE,
        class_weight='balanced', n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE
    ),
    'XGBoost': XGBClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE,
        scale_pos_weight=len(y_train[y_train == 0]) / max(len(y_train[y_train == 1]), 1),
        eval_metric='logloss'
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=RANDOM_STATE,
        class_weight='balanced', verbose=-1
    ),
}

results = []
proba_store = {}

for name, model in models.items():
    print('=' * 60)
    print('Training:', name)
    t0 = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - t0

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    proba_store[name] = y_proba

    row = {
        'Model': name,
        'Train_Time_sec': elapsed,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test, y_proba),
        'Threshold': 0.50,
    }
    results.append(row)
    print(
        f"  Acc={row['Accuracy']:.4f}  P={row['Precision']:.4f}  "
        f"R={row['Recall']:.4f}  F1={row['F1']:.4f}  AUC={row['ROC_AUC']:.4f}  ({elapsed:.2f}s)"
    )

metrics_df = pd.DataFrame(results).sort_values('F1', ascending=False).reset_index(drop=True)
print(metrics_df)

best_model_name = metrics_df.loc[0, 'Model']
best_model = models[best_model_name]
best_f1_default = float(metrics_df.loc[0, 'F1'])
print(f'BEST @ 0.50 by F1: {best_model_name} (F1={best_f1_default:.4f})')


## 8. Threshold tuning on the best model (maximize F1 on test)


In [ ]:
y_scores = proba_store[best_model_name]
threshold_grid = np.arange(0.10, 0.91, 0.01)
tune_rows = []
for thr in threshold_grid:
    y_hat = (y_scores >= thr).astype(int)
    tune_rows.append({
        'Threshold': float(thr),
        'Precision': precision_score(y_test, y_hat, zero_division=0),
        'Recall': recall_score(y_test, y_hat, zero_division=0),
        'F1': f1_score(y_test, y_hat, zero_division=0),
        'Accuracy': accuracy_score(y_test, y_hat),
    })
threshold_df = pd.DataFrame(tune_rows)
best_thr_idx = threshold_df['F1'].idxmax()
selected_threshold = float(threshold_df.loc[best_thr_idx, 'Threshold'])
tuned = threshold_df.loc[best_thr_idx]

print(f'Selected threshold (max F1): {selected_threshold:.2f}')
print('Default 0.50:')
print(metrics_df[metrics_df['Model'] == best_model_name][['Precision','Recall','F1','Accuracy']].to_string(index=False))
print('Optimized:')
print(tuned[['Precision','Recall','F1','Accuracy']])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(threshold_df['Threshold'], threshold_df['Precision'], label='Precision')
ax.plot(threshold_df['Threshold'], threshold_df['Recall'], label='Recall')
ax.plot(threshold_df['Threshold'], threshold_df['F1'], label='F1')
ax.axvline(selected_threshold, color='black', linestyle='--', label=f'best={selected_threshold:.2f}')
ax.set_xlabel('Threshold')
ax.set_ylabel('Score')
ax.set_title(f'Threshold tuning: {best_model_name}')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 9. Save metrics, predictions, and results.json


In [ ]:
# Metrics at default 0.50
metrics_path = OUTPUT_DIR / 'metrics.csv'
metrics_df.to_csv(metrics_path, index=False)

# Predictions for test set (best model)
pred_df = pd.DataFrame({
    'GRID_ID': grid_test.values,
    'City_name': city_test.values,
    'y_true': y_test.values,
    'y_proba_best': y_scores,
    'y_pred_0_50': (y_scores >= 0.50).astype(int),
    'y_pred_tuned': (y_scores >= selected_threshold).astype(int),
    'best_model': best_model_name,
    'tuned_threshold': selected_threshold,
})
# also store all model probabilities
for name, proba in proba_store.items():
    safe = name.lower().replace(' ', '_')
    pred_df[f'proba_{safe}'] = proba

pred_path = OUTPUT_DIR / 'predictions.csv'
pred_df.to_csv(pred_path, index=False)

results = {
    'data_path': str(DATA_PATH),
    'random_state': RANDOM_STATE,
    'test_frac_target': TEST_FRAC,
    'n_total_hexes': int(len(df_pct)),
    'n_train_hexes': int(n_train),
    'n_test_hexes': int(n_test),
    'train_hex_pct': float(n_train / len(df_pct)),
    'test_hex_pct': float(n_test / len(df_pct)),
    'n_train_cities': len(train_cities),
    'n_test_cities': len(test_cities),
    'train_cities': train_cities,
    'test_cities': test_cities,
    'n_features_final': len(final_features),
    'final_features': final_features,
    'removed_vif_features': removed_vif,
    'removed_corr_features': features_to_drop_corr,
    'target': 'Total_Gun_Violence > 0',
    'metrics_at_0_50': metrics_df.to_dict(orient='records'),
    'best_model_at_0_50': best_model_name,
    'best_f1_at_0_50': best_f1_default,
    'threshold_tuning': {
        'best_model': best_model_name,
        'selected_threshold': selected_threshold,
        'precision': float(tuned['Precision']),
        'recall': float(tuned['Recall']),
        'f1': float(tuned['F1']),
        'accuracy': float(tuned['Accuracy']),
    },
}

results_path = OUTPUT_DIR / 'results.json'
with open(results_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2)

print('Wrote:')
print(' ', metrics_path)
print(' ', pred_path)
print(' ', results_path)
print(' ', city_assignment_path)
print()
print('SUMMARY')
print(f'  Train hexes: {n_train:,} | Test hexes: {n_test:,}')
print(f'  Best model @0.50: {best_model_name} | F1={best_f1_default:.4f}')
print(f'  Tuned threshold: {selected_threshold:.2f} | F1={float(tuned["F1"]):.4f}')
